# 06 — Paper Figures & Tables

Loads all Phase 1–3 results and produces publication-grade figures + LaTeX-ready tables for the Overleaf write-up. Run **last**, after Phases 1, 2a, 2b, 3 have all completed.

Outputs:
- `results/figures/paper_*.png` (consistent style, 150 dpi)
- `results/tables/*.tex` (booktabs LaTeX)
- `docs/results_summary.md` (numerical results + per-hypothesis verdict)

Runs in ~5 min on CPU. No GPU needed.

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kvrancic/nlp-project.git'
REPO_DIR = '/content/nlp-project'
if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
!pip install -q 'numpy>=2.0' -e .

for _mod_name in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_mod_name]

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_RESULTS = Path('/content/drive/MyDrive/nlp-project-results')
    # Pull the latest .pt files from Drive in case local /content was wiped.
    for name in ['phase1_features.pt', 'phase2_zhao_baseline.pt',
                 'phase2_ablation.pt', 'phase3_interaction.pt']:
        src = DRIVE_RESULTS / name
        dst = Path('results') / name
        if src.exists() and not dst.exists():
            import shutil; shutil.copy(src, dst)
            print(f'Restored {name} from Drive')
except Exception:
    DRIVE_RESULTS = None

In [ ]:
import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from collections import Counter

from src.config import TARGET_LANGUAGES, SAE_SUBSET_LAYERS, RESULTS_DIR

FIG_DIR = Path('results/figures'); FIG_DIR.mkdir(exist_ok=True, parents=True)
TAB_DIR = Path('results/tables');  TAB_DIR.mkdir(exist_ok=True, parents=True)
DOC_DIR = Path('docs');            DOC_DIR.mkdir(exist_ok=True, parents=True)

# Consistent style across all paper figures
sns.set_theme(style='whitegrid', context='paper', font_scale=1.0)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['font.family'] = 'sans-serif'
LANG_PALETTE = {l: c for l, c in zip(TARGET_LANGUAGES, sns.color_palette('tab10', 5))}

In [ ]:
phase1  = torch.load(RESULTS_DIR / 'phase1_features.pt',     weights_only=False)
phase2a = torch.load(RESULTS_DIR / 'phase2_zhao_baseline.pt', weights_only=False)
phase2b = torch.load(RESULTS_DIR / 'phase2_ablation.pt',      weights_only=False)
phase3  = torch.load(RESULTS_DIR / 'phase3_interaction.pt',   weights_only=False)
print('All phase artifacts loaded.')
print(f'  Phase 1 layers: {phase1["config"]["layers"]}')
print(f'  Phase 2a best (lm, lh, r): ({phase2a["config"]["best_lambda_middle"]}, '
      f'{phase2a["config"]["best_lambda_higher"]}, {phase2a["config"]["best_rank"]})')
print(f'  Phase 2b best k: {phase2b["best_k"]}')
print(f'  Phase 3 best k: {phase3["config"]["best_k"]}')

## 1. Headline figure — H1/H2 main result

In [ ]:
baseline = phase2a['baseline_results']
zhao     = phase2a['zhao_test']
sae      = phase2b['h1_test']
best_k   = phase2b['best_k']

df = pd.DataFrame({
    'unmodified': {l: baseline[l]['accuracy'] for l in TARGET_LANGUAGES},
    f'Zhao SVD':  {l: zhao[l]['accuracy']     for l in TARGET_LANGUAGES},
    f'SAE (k={best_k})': {l: sae[l]['accuracy'] for l in TARGET_LANGUAGES},
})
fig, ax = plt.subplots(figsize=(7.5, 3.5))
df.plot(kind='bar', ax=ax, width=0.8,
        color=['#a0a0a0', '#5a86b5', '#c0392b'])
ax.set_ylabel('MGSM accuracy (250 problems / lang)')
ax.set_xlabel('language')
ax.set_ylim(0, 1.0)
ax.set_title('Headline: SAE-targeted language ablation vs SVD baseline (Gemma 3 4B IT)')
ax.legend(loc='upper right', frameon=True)
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=7, padding=2)
plt.savefig(FIG_DIR / 'paper_fig1_headline.png'); plt.show()

## 2. Phase 1 — feature taxonomy

In [ ]:
# Per-layer probe accuracy + reasoning candidate counts (compact 2-panel)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar([f'L{l}' for l in SAE_SUBSET_LAYERS],
            [phase1['probe_accuracies'][l] for l in SAE_SUBSET_LAYERS],
            color='#5a86b5')
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('Method B: probe accuracy')
axes[0].set_title('Languages are linearly separable in SAE-feature space')
axes[1].bar([f'L{l}' for l in SAE_SUBSET_LAYERS],
            [len(phase1['reasoning_features'][l]) for l in SAE_SUBSET_LAYERS],
            color='#9b6fa3')
axes[1].set_ylabel('# cross-lingual reasoning candidates')
axes[1].set_title('Method C: features active across all 5 langs (>10% of problems)')
plt.savefig(FIG_DIR / 'paper_fig2_phase1.png'); plt.show()

# Method agreement heatmap
jac = pd.DataFrame(phase1['jaccard_AB'])
fig, ax = plt.subplots(figsize=(5, 3))
sns.heatmap(jac, annot=True, fmt='.2f', cmap='magma', vmin=0, vmax=1,
            cbar_kws={'label': 'Jaccard(monolinguality, probe)'}, ax=ax)
ax.set_title('Method agreement: monolinguality (A) vs probe (B), top-50 features')
plt.savefig(FIG_DIR / 'paper_fig3_method_agreement.png'); plt.show()

## 3. Phase 2 — H1/H2/H3 panels

In [ ]:
# H1: accuracy vs k (dev sweep)
h1_sweep = phase2b['h1_sweep']
K_VALUES = phase2b['config']['k_values']
fig, ax = plt.subplots(figsize=(6, 3.5))
for lang in TARGET_LANGUAGES:
    ys = [h1_sweep[k][lang]['accuracy'] for k in K_VALUES]
    ax.plot(K_VALUES, ys, marker='o', color=LANG_PALETTE[lang], label=lang)
ax.axhline(phase2a['baseline_avg'], ls='--', color='grey', label='baseline avg', alpha=0.7)
ax.set_xlabel('k (# LANGUAGE features ablated at L17)')
ax.set_ylabel('MGSM accuracy (50-prompt dev)')
ax.set_title('H1: ablation depth × accuracy')
ax.legend(loc='best')
plt.savefig(FIG_DIR / 'paper_fig4_h1_sweep.png'); plt.show()

In [ ]:
# H2: accuracy × language fidelity scatter
fid = phase2b['fidelity']
fig, ax = plt.subplots(figsize=(5.5, 4))
for lang in TARGET_LANGUAGES:
    color = LANG_PALETTE[lang]
    ax.scatter(fid[lang]['baseline'], baseline[lang]['accuracy'],
               marker='o', s=80, color=color, edgecolor='black', label=f'{lang} baseline')
    ax.scatter(fid[lang]['zhao'],     zhao[lang]['accuracy'],
               marker='s', s=80, color=color, edgecolor='black')
    ax.scatter(fid[lang]['sae'],      sae[lang]['accuracy'],
               marker='^', s=100, color=color, edgecolor='black')
    # connect
    ax.plot([fid[lang]['baseline'], fid[lang]['zhao'], fid[lang]['sae']],
            [baseline[lang]['accuracy'], zhao[lang]['accuracy'], sae[lang]['accuracy']],
            color=color, alpha=0.4, lw=0.5)
from matplotlib.lines import Line2D
method_legend = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='grey',
           markersize=10, label='unmodified', markeredgecolor='black'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='grey',
           markersize=10, label='Zhao SVD',   markeredgecolor='black'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='grey',
           markersize=12, label=f'SAE k={best_k}', markeredgecolor='black'),
]
lang_legend = [Line2D([0], [0], marker='o', color='w', markerfacecolor=LANG_PALETTE[l],
                       markersize=10, label=l, markeredgecolor='black') for l in TARGET_LANGUAGES]
first = ax.legend(handles=method_legend, loc='upper left', title='method', frameon=True)
ax.add_artist(first)
ax.legend(handles=lang_legend, loc='lower right', title='language', frameon=True, ncol=2)
ax.set_xlabel('language fidelity (output_lang == input_lang)')
ax.set_ylabel('MGSM accuracy (250-prompt test)')
ax.set_title('H2: accuracy / fidelity trade-off — Pareto frontier')
plt.savefig(FIG_DIR / 'paper_fig5_h2_pareto.png'); plt.show()

In [ ]:
# H3: layer-wise contribution
h3 = phase2b['h3_results']
fig, ax = plt.subplots(figsize=(6, 3.5))
df_h3 = pd.DataFrame({
    f'L{layer}': {l: h3[layer][l]['accuracy'] for l in TARGET_LANGUAGES}
    for layer in SAE_SUBSET_LAYERS
})
df_h3.T.plot(kind='bar', ax=ax,
             color=[LANG_PALETTE[l] for l in TARGET_LANGUAGES])
ax.set_ylabel('MGSM accuracy (50-prompt dev)')
ax.set_xlabel('SAE layer (k=10 ablation, A∩B selection)')
ax.set_title('H3: per-layer contribution to interference')
ax.axhline(phase2a['baseline_avg'], ls='--', color='grey', alpha=0.7)
ax.legend(loc='best', ncol=5, fontsize=8)
plt.savefig(FIG_DIR / 'paper_fig6_h3_layers.png'); plt.show()

## 4. Phase 3 — interaction

In [ ]:
# (a) Capacity competition violins
rows = []
for lang, recs in phase3['capacity']['summary'].items():
    for r in recs:
        rows.append({'lang': lang, 'mean_delta': r['mean_delta']})
fig, ax = plt.subplots(figsize=(6, 3.2))
sns.violinplot(data=pd.DataFrame(rows), x='lang', y='mean_delta', ax=ax,
               inner='quartile', density_norm='width',
               palette=[LANG_PALETTE[l] for l in TARGET_LANGUAGES])
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Δ activation, ablated − clean (reasoning features)')
ax.set_title('Phase 3 (a): capacity competition at layer 17')
plt.savefig(FIG_DIR / 'paper_fig7_capacity.png'); plt.show()

In [ ]:
# (c) Attention entropy heatmap
attn_df = pd.DataFrame(phase3['attention']['summary'])
heat = attn_df.pivot(index='layer', columns='lang', values='mean_delta_entropy')
fig, ax = plt.subplots(figsize=(5.2, 3.2))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Δ attn entropy (ablated − clean)'}, ax=ax)
ax.set_title('Phase 3 (c): attention entropy shift at last input query')
plt.savefig(FIG_DIR / 'paper_fig8_attention.png'); plt.show()

## 5. LaTeX-ready tables

In [ ]:
# Main results table
ci = phase2b['ci_summary']
rows = []
for lang in TARGET_LANGUAGES:
    rows.append({
        'lang': lang,
        'baseline': baseline[lang]['accuracy'],
        'baseline_ci': ci[lang]['baseline'],
        'zhao': zhao[lang]['accuracy'],
        'zhao_ci': ci[lang]['zhao'],
        'sae': sae[lang]['accuracy'],
        'sae_ci': ci[lang]['sae'],
    })
main_df = pd.DataFrame(rows)

def fmt(x, ci_lo, ci_hi):
    return f'{x:.3f} [{ci_lo:.3f},{ci_hi:.3f}]'

lines = [
    '\\begin{tabular}{lccc}',
    '\\toprule',
    'lang & baseline & Zhao SVD & SAE (k='+str(best_k)+') \\\\',
    '\\midrule',
]
for r in rows:
    lines.append(f"{r['lang']} & {fmt(r['baseline'], *r['baseline_ci'])} & "
                 f"{fmt(r['zhao'], *r['zhao_ci'])} & "
                 f"{fmt(r['sae'], *r['sae_ci'])} \\\\")
avg = main_df[['baseline', 'zhao', 'sae']].mean()
lines.append('\\midrule')
lines.append(f'avg & {avg["baseline"]:.3f} & {avg["zhao"]:.3f} & {avg["sae"]:.3f} \\\\')
lines.append('\\bottomrule')
lines.append('\\end{tabular}')
tex = '\n'.join(lines)
(TAB_DIR / 'main_results.tex').write_text(tex)
print(tex)

In [ ]:
# Causal feature taxonomy table
tag_rows = []
for (lang, feat), v in phase2b['causal_labels'].items():
    tag_rows.append({'lang': lang, 'feature': feat, 'tag': v['tag'],
                     'acc_delta': v['acc_delta'], 'ppl_delta': v['ppl_delta']})
tag_df = pd.DataFrame(tag_rows)
tag_counts = tag_df.groupby(['lang', 'tag']).size().unstack(fill_value=0)
for tag in ['LANGUAGE', 'REASONING', 'SHARED', 'JUNK']:
    if tag not in tag_counts.columns: tag_counts[tag] = 0
tag_counts = tag_counts[['LANGUAGE', 'REASONING', 'SHARED', 'JUNK']]
print('Causal labels per language (top-5 candidates per lang at L17):')
print(tag_counts)
(TAB_DIR / 'causal_labels.tex').write_text(tag_counts.to_latex())
print('\nWritten to results/tables/causal_labels.tex')

## 6. Results summary markdown

In [ ]:
def verdict(h1_avg, baseline_avg, zhao_avg):
    if h1_avg > zhao_avg + 0.005: return 'CONFIRMED — SAE > Zhao'
    if h1_avg >= baseline_avg + 0.005: return 'PARTIAL — SAE > baseline, ≈ Zhao'
    if abs(h1_avg - baseline_avg) < 0.005: return 'NULL — no effect'
    return 'NEGATIVE — SAE < baseline'

h1_avg = float(np.mean([sae[l]['accuracy'] for l in TARGET_LANGUAGES]))
v_h1 = verdict(h1_avg, phase2a['baseline_avg'], phase2a['zhao_avg'])

summary_lines = [
    '# Results Summary — Language-Reasoning Interference via SAEs',
    '',
    '## Headline numbers (250 × 5 MGSM, full test split)',
    '',
    f'- Unmodified Gemma 3 4B IT: **{phase2a["baseline_avg"]:.3f}** avg accuracy',
    f'- Zhao SVD baseline (lm={phase2a["config"]["best_lambda_middle"]}, '
      f'lh={phase2a["config"]["best_lambda_higher"]}, r={phase2a["config"]["best_rank"]}): '
      f'**{phase2a["zhao_avg"]:.3f}** ({phase2a["zhao_avg"]-phase2a["baseline_avg"]:+.3f})',
    f'- SAE-targeted (k={best_k} at L17): **{h1_avg:.3f}** ({h1_avg - phase2a["baseline_avg"]:+.3f})',
    '',
    '## Per-language breakdown',
    '',
    '| lang | baseline | Zhao SVD | SAE | Δ vs baseline |',
    '|------|----------|----------|-----|----------------|',
]
for lang in TARGET_LANGUAGES:
    b, z, s = baseline[lang]['accuracy'], zhao[lang]['accuracy'], sae[lang]['accuracy']
    summary_lines.append(f'| {lang} | {b:.3f} | {z:.3f} | {s:.3f} | {s-b:+.3f} |')

summary_lines += [
    '',
    '## Hypothesis verdicts',
    '',
    f'- **H1** (SAE ablation reproduces Zhao gains): {v_h1}',
]
# H2 verdict from Pareto: SAE-targeted should sit upper-left of the cloud.
fid_avg = {m: float(np.mean([phase2b['fidelity'][l][m] for l in TARGET_LANGUAGES]))
           for m in ['baseline', 'zhao', 'sae']}
summary_lines.append(
    f'- **H2** (better trade-off): mean fidelity baseline={fid_avg["baseline"]:.3f}, '
    f'zhao={fid_avg["zhao"]:.3f}, sae={fid_avg["sae"]:.3f}; '
    f'mean acc ratio (SAE/Zhao)={(h1_avg/max(phase2a["zhao_avg"], 1e-6)):.3f}, '
    f'fidelity ratio={(fid_avg["sae"]/max(fid_avg["zhao"], 1e-6)):.3f}'
)
# H3: which layer dominates
h3_avgs = {layer: phase2b['h3_results'][layer]['avg'] for layer in SAE_SUBSET_LAYERS}
best_layer = max(h3_avgs, key=h3_avgs.get)
summary_lines.append(
    f'- **H3** (middle layers dominate): per-layer dev avg = {h3_avgs}; '
    f'best layer for ablation = {best_layer}'
)

# Phase 3 verdicts
summary_lines += [
    '',
    '## Phase 3 — interaction mechanism',
    '',
]
cap_means = {l: float(np.mean([r['mean_delta'] for r in phase3['capacity']['summary'][l]]))
             for l in TARGET_LANGUAGES}
summary_lines.append(
    f'- (a) capacity competition: per-lang mean Δ activation of reasoning features = {cap_means}'
)
attn_pivot = pd.DataFrame(phase3['attention']['summary']).pivot(
    index='layer', columns='lang', values='mean_delta_entropy')
summary_lines.append(
    f'- (c) attention disruption: per-(layer, lang) Δ entropy =\n\n```\n{attn_pivot.round(4).to_string()}\n```'
)
summary_lines.append(
    f'- (b) circuit attribution: {len(phase3["circuit"]["edges"])} unique downstream effects '
    f'across all (lang, problem) instances. See paper_fig13_circuit_attribution.png.'
)

# Causal labels rollup
tags = Counter(v['tag'] for v in phase2b['causal_labels'].values())
summary_lines += [
    '',
    '## Causal feature labels (top-5 A∩B candidates per lang at L17)',
    '',
    f'- Tag distribution: {dict(tags)}',
    f'- LANGUAGE-confirmed features per language:',
]
for lang in TARGET_LANGUAGES:
    summary_lines.append(f'  - {lang}: {phase2b["confirmed_language"][lang]}')

summary_lines += [
    '',
    '## Files',
    '',
    '- Figures: `results/figures/paper_fig*.png`',
    '- Tables: `results/tables/main_results.tex`, `causal_labels.tex`',
    '- Raw artifacts: `results/phase{1,2_zhao_baseline,2_ablation,3_interaction}.pt`',
]
out = DOC_DIR / 'results_summary.md'
out.write_text('\n'.join(summary_lines))
print(f'Wrote {out}')
print('\n--- summary ---')
print('\n'.join(summary_lines))